# Lecture 4.4 — RunConfig: max_turns, model overrides, workflow_name

**Section 04 — Running Agents, Results & Streaming**

In this notebook you will learn how to control run-level behaviour with `RunConfig`, and how the separate `max_turns` parameter caps how many turns an agent run can take before it gives up.

## 1. Install the OpenAI Agents SDK

This notebook uses the `openai-agents` Python package. The cell below installs it into the current runtime.

The version is pinned so that the examples in this notebook behave exactly as shown. If the package is already installed in this session, pip will confirm it is present and move on quickly.

In [12]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.2 -q

## 2. Set up your OpenAI API key

This notebook uses **Google Colab Secrets** to store your API key, so it never appears in plain text in the notebook.

**Steps to add your key in Colab:**
1. Click the key icon (🔑) in the left sidebar to open **Secrets**.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your key as the value.
4. Toggle **Notebook access** on for this notebook.
5. Run the cell below.

**Running locally instead of Colab?** Set the `OPENAI_API_KEY` environment variable in your terminal before launching Jupyter, for example: `export OPENAI_API_KEY="sk-..."`. The `userdata.get(...)` call below is Colab-specific and will need to be replaced with a direct read from `os.environ`.

In [13]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 3. Declare the model name

`MODEL_NAME` is declared once here and used everywhere an `Agent` in this notebook needs a model. Changing this single variable updates the model used across the entire notebook, so you never have to hunt down hardcoded model strings later.

In [14]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## 4. Imports

| Import | Purpose |
|---|---|
| `uuid` | Generates a unique `group_id` value for linking related traces later in this notebook. |
| `Reasoning` (from `openai.types.shared`) | Configures reasoning effort on `ModelSettings`, used in a couple of the agents below. |
| `Agent` | Defines an agent: its name, instructions, model, and tools. |
| `MaxTurnsExceeded` | The exception raised when a run exceeds its `max_turns` limit. |
| `ModelSettings` | Per-agent or per-run model configuration (reasoning effort, verbosity, temperature, and so on). |
| `RunConfig` | The dataclass that controls run-level behaviour: model overrides, tracing, and more. This is the main subject of this lecture. |
| `Runner` | Executes an agent run. Every run in this notebook uses `await Runner.run(...)`. |
| `ToolExecutionConfig` | SDK-side tool concurrency settings, mentioned as one of the `RunConfig` fields but not used directly in this notebook. |
| `function_tool` | Decorator that turns a plain Python function into a tool an agent can call. |

`MaxTurnsExceeded`, `RunConfig`, and `ToolExecutionConfig` are all importable directly from the top-level `agents` package, alongside `Agent`, `ModelSettings`, `Runner`, and `function_tool`.

In [15]:
import uuid

from openai.types.shared import Reasoning

from agents import (
    Agent,
    MaxTurnsExceeded,
    ModelSettings,
    RunConfig,
    Runner,
    ToolExecutionConfig,
    function_tool,
)

## 5. What is RunConfig?

Every call to `Runner.run()` executes one agent, but that agent might hand off to other agents, call tools, and generate multiple traces along the way. `RunConfig` is how you control settings for the **entire run**, rather than for a single agent.

`RunConfig` groups its fields into four areas:

**1. Model and execution**
- `model` — override the model for every agent in the run
- `model_provider` — resolves string model names (defaults to OpenAI)
- `model_settings` — override model settings (reasoning, verbosity, temperature, and so on) across every agent
- `tool_execution` — SDK-side concurrency settings for local tool calls (covered in Lecture 3.7)

**2. Tracing**
- `workflow_name` — the label for this run in the OpenAI traces dashboard
- `trace_id`, `group_id`, `trace_metadata` — identify and link traces
- `tracing_disabled` — turn tracing off entirely
- `trace_include_sensitive_data` — redact tool and model I/O from trace content

**3. Guardrails** *(preview, covered fully in Section 5)*
- `input_guardrails`, `output_guardrails` — run-level guardrails applied on top of any agent-level guardrails

**4. Handoff settings** *(covered fully in Section 5)*
- `handoff_input_filter` — a global filter applied to every handoff in the run
- `reasoning_item_id_policy` — controls whether reasoning item IDs are preserved or stripped between turns

### The one thing that trips people up: max_turns is NOT on RunConfig

It is tempting to assume `max_turns` lives inside `RunConfig`, since it is also a run-level setting. It does not. `max_turns` is a **direct parameter of `Runner.run()`**, passed alongside `run_config`, not nested inside it:

```python
result = await Runner.run(
    agent,
    input,
    max_turns=5,            # direct parameter
    run_config=RunConfig(   # separate parameter
        workflow_name="...",
    ),
)
```

Keep that distinction in mind as you work through the rest of this notebook.

## 6. max_turns — controlling turn limits

A **turn** is one invocation of the model within a run (including any tool calls the model makes on that turn). By default, `Runner.run()` allows up to `DEFAULT_MAX_TURNS` turns, which is **10**, before it gives up and raises `MaxTurnsExceeded`.

The agent below is instructed to call a tool three times in sequence before answering, which makes it easy to see the turn limit in action.

`result.raw_responses` holds one entry per turn the run actually took, one model response per round trip. len(result.raw_responses) is not an estimate, it's the literal turn count for this run. Run the cell and see what number comes back before reading further: if the model called get_step three times sequentially before answering, expect 4.

When `MaxTurnsExceeded` is raised, the exception itself only carries a `message`. The agent that was running when the limit was hit is available on `e.run_data.last_agent`, not directly on the exception. `run_data` is a `RunErrorDetails` object the SDK attaches to the exception right before raising it, and `last_agent` is one of its fields.

In [16]:
@function_tool
def get_step(step_number: int) -> str:
    """Returns a step in a multi-step process.

    Args:
        step_number: The step number to retrieve.
    """
    steps = {
        1: "First, gather requirements.",
        2: "Second, design the solution.",
        3: "Third, implement and test.",
    }
    return steps.get(step_number, "No more steps.")


agent = Agent(
    name="Process Agent",
    instructions=(
        "You are a process assistant. Use get_step to "
        "walk through steps 1, 2, and 3 before answering."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[get_step],
)

# Default: max_turns=10 (DEFAULT_MAX_TURNS from run_config.py)
result = await Runner.run(
    agent,
    "Walk me through the process.",
)
print("Default (max_turns=10):\n", result.final_output[:100])
print("Turns taken:", len(result.raw_responses))

Default (max_turns=10):
 1. Gather requirements.  
2. Design the solution.  
3. Implement and test.
Turns taken: 4


In [17]:
# Restrict to 2 turns. This is not enough for three tool calls
# plus a final answer, so it will trigger MaxTurnsExceeded.
try:
    result_short = await Runner.run(
        agent,
        "Walk me through the process.",
        max_turns=2,
    )
    print("Short run:", result_short.final_output[:100])
except MaxTurnsExceeded as e:
    print(f"MaxTurnsExceeded: Error is {e}. Last agent was {e.run_data.last_agent.name}")

MaxTurnsExceeded: Error is Max turns (2) exceeded. Last agent was Process Agent


## 7. RunConfig.model — run-level model override

`RunConfig.model` overrides the model on **every agent in the run**, regardless of what model each individual `Agent` was defined with. This is useful for A/B testing a workflow against a different model without editing every agent definition.

`fast_agent` below is defined with `MODEL_NAME`, but the `RunConfig` passed into `Runner.run()` overrides it with `"gpt-5.5"` for this run only. `RunConfig.model_settings` works the same way: any non-null value you set there overrides the corresponding per-agent setting.

Don't just take that on faith, though. After running this cell, open [platform.openai.com/traces](https://platform.openai.com/traces) and look for the trace named "Model override demo." Open it, click into its `POST /v1/responses` span, and check the `Model` field in the Properties panel. It should read `gpt-5.5`, not `gpt-5.4-mini`, confirming the override reached the actual API request.

In [19]:
fast_agent = Agent(
    name="Fast Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

# Override model for entire run. fast_agent's own model is ignored.
result = await Runner.run(
    fast_agent,
    "What is 2 + 2?",
    run_config=RunConfig(
        model="gpt-5.5",
        model_settings=ModelSettings(
            reasoning=Reasoning(effort="low"),
            verbosity="low",
        ),
        workflow_name="Model override demo",
    ),
)
print("Result (using gpt-5.5):", result.final_output)

Result (using gpt-5.5): 4


## 8. workflow_name, group_id, and trace_metadata

Three tracing fields work together to make a run identifiable and linkable in the OpenAI traces dashboard:

- **`workflow_name`** is the label shown for this run. The default, `"Agent workflow"`, tells you nothing when you are scanning a dashboard full of runs. Always set something specific.
- **`group_id`** links multiple separate `Runner.run()` calls together as one conversation or session. Any stable identifier works: a chat thread ID, a session ID, or (as below) a UUID generated for this demo.
- **`trace_metadata`** attaches an arbitrary dictionary of key-value pairs to the trace, useful for things like `user_id`, `environment`, or `tenant_id`.

Below, two separate runs share the same `group_id`. Each run still gets its own row in the traces list, with its own duration and timestamp: `group_id` is not a visual merge. What it gives you is a **linking key**. Filter or search by that `group_id` value in the dashboard and both runs come up together, letting you reconstruct the full conversation from separate traces.

In [20]:
session_id = str(uuid.uuid4())

result1 = await Runner.run(
    fast_agent,
    "Tell me a joke.",
    run_config=RunConfig(
        workflow_name="Joke generation workflow",
        group_id=session_id,
        trace_metadata={
            "user_id": "u001",
            "environment": "demo",
        },
    ),
)
print("Turn 1:", result1.final_output[:80])

result2 = await Runner.run(
    fast_agent,
    result1.to_input_list() + [
        {"role": "user", "content": "Tell me another one."}
    ],
    run_config=RunConfig(
        workflow_name="Joke generation workflow",
        group_id=session_id,
        trace_metadata={
            "user_id": "u001",
            "environment": "demo",
        },
    ),
)
print("Turn 2:", result2.final_output[:80])
print("Both runs linked by group_id:", session_id)

Turn 1: Why don’t skeletons fight each other?

Because they don’t have the guts.
Turn 2: Sure — why did the scarecrow win an award?

Because he was outstanding in his fi
Both runs linked by group_id: 8759ea88-ebd8-45d2-b86d-1ffaf0e79ed7


## 9. tracing_disabled — turning tracing off entirely

Setting `tracing_disabled=True` disables all tracing for that run. No spans are created and no data is sent to the traces dashboard at all.

This is useful during local development when you do not need trace data cluttering your dashboard, or in contexts where you would rather not send any run data to the dashboard at all.

After running this cell, open [platform.openai.com/traces](https://platform.openai.com/traces) and refresh. You will not see a workflow named "No trace run" anywhere in the list, confirming that `tracing_disabled=True` suppressed the trace entirely rather than just hiding it from view.

In [21]:
result = await Runner.run(
    fast_agent,
    "What is the capital of France?",
    run_config=RunConfig(
        workflow_name="No trace run",
        tracing_disabled=True,
    ),
)
print("Result (no tracing):", result.final_output)
print("Note: this run does not appear in the traces dashboard.")

Result (no tracing): Paris.
Note: this run does not appear in the traces dashboard.


## 10. trace_include_sensitive_data — redacting trace content

`trace_include_sensitive_data` is a lighter touch than `tracing_disabled`. When set to `False`, spans are still created and the run's structure is still visible in the dashboard, but the actual content of tool inputs, tool outputs, and LLM generations is excluded from what gets sent to OpenAI's backend.

The default is `True`, and it can also be controlled globally with the `OPENAI_AGENTS_TRACE_INCLUDE_SENSITIVE_DATA` environment variable. Use `False` when a run touches PII, financial data, or anything else you would not want sitting in a trace log.

After running this cell, open [platform.openai.com/traces](https://platform.openai.com/traces) and find the "Sensitive data run" trace. The span tree (task, agent, turn, `POST /v1/responses`) is fully present, confirming spans are still created, and the workflow name and any `trace_metadata` are still visible. But clicking into the `POST /v1/responses` row shows **"Could not fetch Response — An error occurred while fetching log details"** instead of the actual prompt and completion. That error is expected: because `trace_include_sensitive_data=False`, the response content was never sent to OpenAI's backend, so the dashboard has nothing to retrieve. It is proof the redaction worked, not a sign that something broke.

In [22]:
result = await Runner.run(
    fast_agent,
    "Process this document: Reference ID 8734-A.",
    run_config=RunConfig(
        workflow_name="Sensitive data run",
        trace_metadata={
            "user_id": "u001",
            "environment": "demo",
        },
        trace_include_sensitive_data=False,
    ),
)
print("Result:", result.final_output[:80])
print(
    "Note: spans are created but tool I/O and LLM "
    "generations are excluded from trace content."
)

Result: I can help, but I don’t have the document itself.

Please upload or paste the co
Note: spans are created but tool I/O and LLM generations are excluded from trace content.


## 11. RunConfig.model_settings — run-level settings override

`RunConfig.model_settings` merges with each agent's own `model_settings` using the same `.resolve()` mechanism from Lecture 2.2: only the fields you explicitly set on the `RunConfig` side override the agent's values, everything else is left alone.

`creative_base` below is defined with `temperature=0.0` for precise, repeatable output. The second run overrides just the temperature at the `RunConfig` level, without touching the agent definition, to get a more varied response. Both runs are given their own `workflow_name` so each shows up as its own labeled trace.

**Note on GPT-5 models and temperature:** reasoning models have historically rejected an explicit `temperature` value with a `400 Unsupported parameter` error, since they are effectively locked to `temperature=1` internally. `gpt-5.4-mini` does accept an explicit `temperature`, so this cell runs successfully, but if you swap in a different reasoning model later and see that error, this is why.

To see direct proof the override reached the API, rather than just comparing two pieces of writing, open [platform.openai.com/traces](https://platform.openai.com/traces) after running this cell and open each trace's `POST /v1/responses` span. The **Properties → Configuration** panel shows the exact request parameters used for that call, including `Temperature`. "Precise Run Demo" should show `Temperature: 0.00`, and "Creative Run Demo" should show `Temperature: 1.40`, confirming the override took effect independently of anything you can infer from the output text.

In [23]:
creative_base = Agent(
    name="Base Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        temperature=0.0,
    ),
)

result_precise = await Runner.run(
    creative_base,
    "Write a one-sentence tagline for a coffee shop.",
    run_config=RunConfig(
        workflow_name="Precise Run Demo",
    ),
)
print("Precise (temp=0.0):", result_precise.final_output)

result_creative = await Runner.run(
    creative_base,
    "Write a one-sentence tagline for a coffee shop.",
    run_config=RunConfig(
        workflow_name="Creative Run Demo",
        model_settings=ModelSettings(temperature=1.4),
    ),
)
print("Creative (temp=1.4 via RunConfig):", result_creative.final_output)

Precise (temp=0.0): Freshly brewed coffee, warm smiles, and a cozy place to linger.
Creative (temp=1.4 via RunConfig): Freshly brewed comfort, one cup at a time.


## 12. Combining max_turns and RunConfig together

Since `max_turns` and `run_config` are separate keyword arguments to `Runner.run()`, there is nothing stopping you from passing both in the same call. This is the normal way most production runs look: a sensible turn limit alongside a properly named, traceable `RunConfig`.

In [24]:
result = await Runner.run(
    agent,
    "Walk me through the process briefly.",
    max_turns=5,
    run_config=RunConfig(
        workflow_name="Combined config demo",
        group_id="demo-session-001",
        tracing_disabled=False,
    ),
)
print("Combined result:", result.final_output[:100])

Combined result: 1. Gather requirements.  
2. Design the solution.  
3. Implement and test.


## 13. RunConfig field reference

The table below covers every field on `RunConfig`, including a few not demonstrated above. `max_turns` is included at the bottom as a reminder that it is a `Runner.run()` parameter, not a `RunConfig` field.

| Field | Type | Default | Description |
|---|---|---|---|
| `model` | `str \| Model \| None` | `None` | Override the model for the entire run |
| `model_provider` | `ModelProvider` | `MultiProvider()` | Resolves string model names |
| `model_settings` | `ModelSettings \| None` | `None` | Override per-agent model settings |
| `tool_execution` | `ToolExecutionConfig \| None` | `None` | SDK-side tool concurrency (Lecture 3.7) |
| `workflow_name` | `str` | `"Agent workflow"` | Trace name shown in the dashboard. Always set this |
| `trace_id` | `str \| None` | `None` | Custom trace ID. Auto-generated if not set |
| `group_id` | `str \| None` | `None` | Links multiple traces from the same conversation |
| `trace_metadata` | `dict \| None` | `None` | Arbitrary metadata attached to the trace |
| `tracing_disabled` | `bool` | `False` | Disables all tracing for the run |
| `tracing` | `TracingConfig \| None` | `None` | Overrides trace export settings (Lecture 6.1) |
| `trace_include_sensitive_data` | `bool` | `True` | Include tool/LLM I/O in traces |
| `input_guardrails` | `list \| None` | `None` | Run-level input guardrails (Section 5) |
| `output_guardrails` | `list \| None` | `None` | Run-level output guardrails (Section 5) |
| `handoff_input_filter` | `callable \| None` | `None` | Global handoff filter (Section 5) |
| `nest_handoff_history` | `bool` | `False` | Opt-in beta: collapse prior history before a handoff (Section 5) |
| `handoff_history_mapper` | `callable \| None` | `None` | Custom transcript-to-input mapping for handoffs (Section 5) |
| `reasoning_item_id_policy` | `str \| None` | `None` | `"preserve"` or `"omit"` for reasoning item IDs |
| `session_input_callback` | `callable \| None` | `None` | Customizes how new input merges with session history |
| `call_model_input_filter` | `callable \| None` | `None` | Hook to edit input immediately before each model call |
| `tool_error_formatter` | `callable \| None` | `None` | Customizes tool error messages returned to the model |
| `tool_not_found_behavior` | `str` | `"raise_error"` | `"raise_error"` or `"return_error_to_model"` |
| `session_settings` | `SessionSettings \| None` | `None` | Overrides session settings |
| `sandbox` | `SandboxRunConfig \| None` | `None` | Sandbox runtime configuration |

**Not on this table:** `max_turns`. It is a direct `Runner.run()` parameter:

```python
result = await Runner.run(
    agent, input,
    max_turns=5,
    run_config=RunConfig(...),
)
```